<a href="https://colab.research.google.com/github/miffylim2308/ADALL_github/blob/main/ADALL_My_Template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- Ctrl + / to remark/unremark
- Ctrl + ] to ident right
- Ctrl + [ to indet left

#**Session 1: From business problem to clean dataset**

#**Chapter 1. Setup and load libraries**

In [1]:
# Core Libraries
import os
import shutil
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# Modelling libraries and preprocessing
from sklearn.model_selection import train_test_split, ShuffleSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

# Make wide tables easier to read in Colab
pd.set_option('display.max_columns', 100)

#**Chapter 2. Load the dataset**

In [2]:
#Load dataset from github
github_raw_url = 'https://raw.githubusercontent.com/rq-goh/ADALL_github/refs/heads/main/laptop_prices_2024_sgd_TL.csv'

df = pd.read_csv(github_raw_url)

print('Dataset loaded successfully.')
print('Shape:', df.shape)
display(df.head())

#Load dataset from kaggle
#path = kagglehub.dataset_download("devansodariya/student-performance-data")
#print("Downloaded to:", path)
#print(os.listdir(path))
#df = pd.read_csv(os.path.join(path, "student_data.csv"))

# This cell loads the laptop price dataset directly from GitHub.
# The GitHub raw URL points to the actual CSV file, not the normal GitHub preview page.
# pd.read_csv() reads the CSV file and stores it as a pandas DataFrame called df.
# df.shape shows the number of rows and columns in the dataset.
# df.head() displays the first 5 rows so we can quickly check that the data loaded correctly.

Dataset loaded successfully.
Shape: (1000, 15)


,Unnamed: 0,Brand,Model,CPU,GPU,RAM_GB,Storage_Type,Storage_GB,Touchscreen,Weight_kg,Screen_Size_inch,Discount_percent,Price_SGD,Brand_Discount,Member_Discount
0,0,Acer,Aspire 5,Intel i9-14900HK,NVIDIA RTX 4070,64,SSD,256,False,1.56,16.0,5.28,3207.60,80,160.38
1,1,Acer,Nitro 5,AMD Ryzen 9 8900HX,AMD Radeon 780M,32,SSD,1024,True,1.45,14.0,6.01,2568.40,80,179.79
2,2,Acer,Nitro 5,AMD Ryzen 5 8600H,NVIDIA RTX 4050,32,SSD,2048,False,1.34,14.0,6.56,2050.80,80,143.56
3,3,Acer,TravelMate P6,Intel Core Ultra 7 15500H,NVIDIA RTX 4060,16,SSD,4096,True,1.18,13.3,4.62,2477.59,80,173.43
4,4,Acer,Predator Helios 300,Intel i7-14800H,NVIDIA RTX 4070,8,SSD,1024,True,1.31,14.0,4.81,2626.40,80,183.85


#**Chapter 2a. Set up OpenAI**

In [34]:
# Default method: copy the prompt into the chatbot manually.
RUN_API_CELLS = True
OPENAI_MODEL = 'gpt-5.4-nano'
client = None

# Only use this section if your tutor has asked you to call the API from Colab.
# You must first save OPENAI_API_KEY in Colab Secrets.

if RUN_API_CELLS == True:
    from google.colab import userdata
    from openai import OpenAI

    api_key = userdata.get('OPENAI_API_KEY')
    client = OpenAI(api_key=api_key)
    print('OpenAI client is ready.')
else:
    print('Manual chatbot mode. Copy the prompts when they appear.')

# This cell prepares the notebook to use the OpenAI API, if needed.
# RUN_API_CELLS controls whether the notebook uses the API or manual chatbot mode.
# If RUN_API_CELLS is True, the notebook will try to connect to OpenAI using an API key.
# If RUN_API_CELLS is False, students can copy the prompt and paste it into ChatGPT manually.
# OPENAI_MODEL stores the model name that will be used later when sending prompts.
# client starts as None first, then becomes an OpenAI client after the API key is loaded.
# userdata.get('OPENAI_API_KEY') reads the API key saved in Colab Secrets.
# OpenAI(api_key=api_key) creates the connection object used to call the API.


OpenAI client is ready.


In [35]:
# TESTING THE PROMPT
prompt_test = f"""
I am testing the prompt generator.
"""
print('=== Prompt to send ===')
print(prompt_test[:2000])

if client is not None:
    response_test = client.responses.create(
        model=OPENAI_MODEL,
        input=prompt_test
    )
    print('\n=== LLM response ===')
    print(response_test.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')

=== Prompt to send ===

I am testing the prompt generator.


=== LLM response ===
Great—I'm ready to help test the prompt generator.

What should the generator produce? For example, tell me:
1) the topic/domain,  
2) the format you want (e.g., JSON, bullet points, a specific template), and  
3) any constraints (length, tone, target audience).


#**Chapter 3. LLM-assisted problem framing**

In [4]:
# PROMPT
prompt1 = f"""
You are an expert data scientist with experience in tree-based regression models.
Help me translate this business problem into a modelling objective.

Business problem: A refurbished laptop seller wants to price laptops fairly and consistently.
Dataset context: The dataset contains laptop specifications and price in SGD.

Please answer:
What should the modelling objective be?
What is the most meaningful target column?
Which metric would be easiest to explain to business users?
Who are the main stakeholders?
What are three risks or pitfalls?
"""
print('=== Prompt to send ===')
print(prompt1[:2000])

if client is not None:
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=prompt1
    )
    print('\n=== LLM response ===')
    print(response.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')


=== Prompt to send ===

You are an expert data scientist with experience in tree-based regression models.
Help me translate this business problem into a modelling objective.

Business problem: A refurbished laptop seller wants to price laptops fairly and consistently.
Dataset context: The dataset contains laptop specifications and price in SGD.

Please answer:
What should the modelling objective be?
What is the most meaningful target column?
Which metric would be easiest to explain to business users?
Who are the main stakeholders?
What are three risks or pitfalls?


=== LLM response ===
### 1) What should the modelling objective be?
Build a regression model that **predicts the selling price (in SGD)** of a refurbished laptop from its specifications, so the seller can:
- **Price consistently** across different models/conditions/configurations
- **Reduce manual pricing bias/variance**
- Use predictions as a “fair baseline,” optionally adjusted for margins and inventory considerations

Fo

#**Chapter 4. Quick dataset inspection before using an LLM**

In [5]:
print('Shape:', df.shape)

display(df.head())

print('\nInfo:')
display(df.info())

print('\nColumns:', df.columns.tolist())

print("\nDescribe:")
display(df.describe(include='all').transpose())

missing_values = df.isnull().sum()
print("Missing values:\n", missing_values)

duplicate_count = df.duplicated().sum()
print("\nDuplicate count:", duplicate_count)

Shape: (1000, 15)


,Unnamed: 0,Brand,Model,CPU,GPU,RAM_GB,Storage_Type,Storage_GB,Touchscreen,Weight_kg,Screen_Size_inch,Discount_percent,Price_SGD,Brand_Discount,Member_Discount
0,0,Acer,Aspire 5,Intel i9-14900HK,NVIDIA RTX 4070,64,SSD,256,False,1.56,16.0,5.28,3207.60,80,160.38
1,1,Acer,Nitro 5,AMD Ryzen 9 8900HX,AMD Radeon 780M,32,SSD,1024,True,1.45,14.0,6.01,2568.40,80,179.79
2,2,Acer,Nitro 5,AMD Ryzen 5 8600H,NVIDIA RTX 4050,32,SSD,2048,False,1.34,14.0,6.56,2050.80,80,143.56
3,3,Acer,TravelMate P6,Intel Core Ultra 7 15500H,NVIDIA RTX 4060,16,SSD,4096,True,1.18,13.3,4.62,2477.59,80,173.43
4,4,Acer,Predator Helios 300,Intel i7-14800H,NVIDIA RTX 4070,8,SSD,1024,True,1.31,14.0,4.81,2626.40,80,183.85



Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        1000 non-null   int64  
 1   Brand             1000 non-null   object 
 2   Model             1000 non-null   object 
 3   CPU               1000 non-null   object 
 4   GPU               1000 non-null   object 
 5   RAM_GB            1000 non-null   int64  
 6   Storage_Type      1000 non-null   object 
 7   Storage_GB        1000 non-null   int64  
 8   Touchscreen       1000 non-null   bool   
 9   Weight_kg         1000 non-null   float64
 10  Screen_Size_inch  1000 non-null   float64
 11  Discount_percent  1000 non-null   float64
 12  Price_SGD         1000 non-null   float64
 13  Brand_Discount    1000 non-null   int64  
 14  Member_Discount   1000 non-null   float64
dtypes: bool(1), float64(5), int64(4), object(5)
memory usage: 110.5+ KB


None


Columns: ['Unnamed: 0', 'Brand', 'Model', 'CPU', 'GPU', 'RAM_GB', 'Storage_Type', 'Storage_GB', 'Touchscreen', 'Weight_kg', 'Screen_Size_inch', 'Discount_percent', 'Price_SGD', 'Brand_Discount', 'Member_Discount']

Describe:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Unnamed: 0,1000.0,NaN,NaN,NaN,499.5,288.819436,0.0,249.75,499.5,749.25,999.0
Brand,1000,6,Asus,177,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Model,1000,30,Predator Helios 300,48,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CPU,1000,10,Intel i5-14600H,114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GPU,1000,9,NVIDIA RTX 4070,268,NaN,NaN,NaN,NaN,NaN,NaN,NaN
RAM_GB,1000.0,NaN,NaN,NaN,53.128,44.413288,8.0,16.0,32.0,64.0,128.0
Storage_Type,1000,1,SSD,1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Storage_GB,1000.0,NaN,NaN,NaN,1505.024,1380.203919,256.0,512.0,1024.0,2048.0,4096.0
Touchscreen,1000,2,False,505,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Weight_kg,1000.0,NaN,NaN,NaN,2.03656,0.746477,1.0,1.34,1.97,2.68,3.5


Missing values:
 Unnamed: 0          0
Brand               0
Model               0
CPU                 0
GPU                 0
RAM_GB              0
Storage_Type        0
Storage_GB          0
Touchscreen         0
Weight_kg           0
Screen_Size_inch    0
Discount_percent    0
Price_SGD           0
Brand_Discount      0
Member_Discount     0
dtype: int64

Duplicate count: 0


#**Chapter 5. First LLM touchpoint: send only a small preview**

In [6]:
data_preview = df.head(10).to_string()
print(data_preview[:1500])

   Unnamed: 0 Brand                Model                        CPU              GPU  RAM_GB Storage_Type  Storage_GB  Touchscreen  Weight_kg  Screen_Size_inch  Discount_percent  Price_SGD  Brand_Discount  Member_Discount
0           0  Acer             Aspire 5           Intel i9-14900HK  NVIDIA RTX 4070      64          SSD         256        False       1.56              16.0              5.28    3207.60              80           160.38
1           1  Acer              Nitro 5         AMD Ryzen 9 8900HX  AMD Radeon 780M      32          SSD        1024         True       1.45              14.0              6.01    2568.40              80           179.79
2           2  Acer              Nitro 5          AMD Ryzen 5 8600H  NVIDIA RTX 4050      32          SSD        2048        False       1.34              14.0              6.56    2050.80              80           143.56
3           3  Acer        TravelMate P6  Intel Core Ultra 7 15500H  NVIDIA RTX 4060      16          SSD       

In [7]:
preview_prompt = f"""
Here are the first 10 rows of a laptop pricing dataset:

{data_preview}

Questions:
1. What does each row appear to represent?
2. Which column is likely the target for a price prediction model?
3. What are 3 possible data quality checks we should perform before modelling?

Keep the answer short and practical.
"""
print('=== Prompt to send ===')
print(preview_prompt[:2000])

if client is not None:
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=preview_prompt
    )
    print('\n=== LLM response ===')
    print(response.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')

=== Prompt to send ===

Here are the first 10 rows of a laptop pricing dataset:

   Unnamed: 0 Brand                Model                        CPU              GPU  RAM_GB Storage_Type  Storage_GB  Touchscreen  Weight_kg  Screen_Size_inch  Discount_percent  Price_SGD  Brand_Discount  Member_Discount
0           0  Acer             Aspire 5           Intel i9-14900HK  NVIDIA RTX 4070      64          SSD         256        False       1.56              16.0              5.28    3207.60              80           160.38
1           1  Acer              Nitro 5         AMD Ryzen 9 8900HX  AMD Radeon 780M      32          SSD        1024         True       1.45              14.0              6.01    2568.40              80           179.79
2           2  Acer              Nitro 5          AMD Ryzen 5 8600H  NVIDIA RTX 4050      32          SSD        2048        False       1.34              14.0              6.56    2050.80              80           143.56
3           3  Acer        Trav

#<font color='red'>**Chapter 7. Build the payload text**</font>

In [8]:
# Build payload text step by step.
# Payload text is a short profile of the dataset.
# It is safer and smaller than sending the full dataset to the LLM.

payload_text = ''

# 0. Business objective
payload_text += '=== BUSINESS OBJECTIVE ===\n'
payload_text += 'Business Goal: Predict laptop prices based on laptop specifications.\n'
payload_text += 'Questions:\n'
payload_text += '- Which features influence laptop price?\n'
payload_text += '- What data quality issues exist?\n'
payload_text += '- What preprocessing is required?\n\n'

# 1. Shape
payload_text += '=== SHAPE ===\n'
payload_text += 'Rows: ' + str(df.shape[0]) + '\n'
payload_text += 'Columns: ' + str(df.shape[1]) + '\n\n'

# 2. Sample records
payload_text += '=== SAMPLE RECORDS ===\n'
payload_text += df.head(5).to_string(index=False)
payload_text += '\n\n'

# 3. Column names and data types
payload_text += '=== COLUMNS AND DATA TYPES ===\n'
payload_text += df.dtypes.to_string()
payload_text += '\n\n'

# 4. Numeric summary
payload_text += '=== NUMERIC SUMMARY ===\n'
numeric_summary = df.describe(include='number').round(2)
payload_text += numeric_summary.to_string()
payload_text += '\n\n'

# 5. Null summary
payload_text += "=== NULL SUMMARY ===\n"
null_summary = (
    df.isna().sum().to_frame("null_count")
    .assign(null_pct=lambda x: x["null_count"]/len(df))
)
payload_text += null_summary.to_string()
payload_text += '\n\n'

# 6. Missing values
payload_text += '=== MISSING VALUES ===\n'
missing_table = pd.DataFrame()
missing_table['missing_count'] = df.isna().sum()
missing_table['missing_pct'] = (df.isna().sum() / len(df) * 100).round(2)
payload_text += missing_table.to_string()
payload_text += '\n\n'

# 7. Unique values per column
payload_text += '=== UNIQUE VALUES PER COLUMN ===\n'
unique_table = pd.DataFrame()
unique_table['unique_count'] = df.nunique(dropna=False)
payload_text += unique_table.to_string()
payload_text += '\n\n'

# 8 Correlation between numeric columns
payload_text += '=== CORRELATION BETWEEN NUMERIC COLUMNS ===\n'
correlation_table = df.corr(numeric_only=True).round(2)
payload_text += correlation_table.to_string()
payload_text += '\n\n'

# 9. Top 10 values for categorical columns
payload_text += '=== TOP 10 VALUES FOR CATEGORICAL COLUMNS ===\n'

categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()

if len(categorical_columns) == 0:
    payload_text += 'No categorical columns found.\n'
else:
    for col in categorical_columns:
        payload_text += '\nColumn: ' + col + '\n'
        payload_text += df[col].value_counts(dropna=False).head(10).to_string()
        payload_text += '\n'

payload_text += '\n'

# 10. Skewness of numeric columns
payload_text += '=== SKEWNESS OF NUMERIC COLUMNS ===\n'
skew_table = df.select_dtypes(include='number').skew().round(2)
payload_text += skew_table.to_string()
payload_text += '\n\n'

# 11. Outlier summary
payload_text += '=== OUTLIER SUMMARY (IQR METHOD) ===\n'

numeric_columns = df.select_dtypes(include='number').columns

for column in numeric_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = ((df[column] < lower_bound) | (df[column] > upper_bound)).sum()
    outlier_pct = round(outlier_count / len(df) * 100, 2)

    payload_text += f'{column}: {outlier_count} ({outlier_pct}%)\n'

payload_text += '\n'

# 12. Leakage scan: columns with all unique values
payload_text += "=== POSSIBLE LEAKAGE COLUMNS (UNIQUE FOR EACH ROW) ===\n"
leak_cols = df.columns[df.nunique() == len(df)]
payload_text +=str(list(leak_cols))
payload_text +="\n\n"

# 13. Simple warning checks
payload_text += '=== SIMPLE WARNING CHECKS ===\n'

id_like_columns = []
constant_columns = []

for col in df.columns:
    unique_count = df[col].nunique(dropna=False)

    if unique_count == len(df):
        id_like_columns.append(col)

    if unique_count <= 1:
        constant_columns.append(col)
payload_text += 'Possible ID-like columns: ' + str(id_like_columns) + '\n'
payload_text += 'Constant columns: ' + str(constant_columns) + '\n'
duplicate_count = df.duplicated().sum()
duplicate_pct = round(duplicate_count / len(df) * 100, 2)
payload_text += (
    f'Duplicate rows: {duplicate_count} ({duplicate_pct}%)\n'
)

print(payload_text)

# This cell builds a short dataset profile called payload_text.
# Instead of sending the full dataset to the LLM, we send summary information only.
# The payload includes shape, column types, numeric summary, missing values, unique counts, correlations, and common category values.
# It also adds simple warning checks for possible ID-like columns, constant columns, and duplicate rows.
# This helps the LLM comment on data readiness without needing every row of the dataset.

=== BUSINESS OBJECTIVE ===
Business Goal: Predict laptop prices based on laptop specifications.
Questions:
- Which features influence laptop price?
- What data quality issues exist?
- What preprocessing is required?

=== SHAPE ===
Rows: 1000
Columns: 15

=== SAMPLE RECORDS ===
 Unnamed: 0 Brand               Model                       CPU             GPU  RAM_GB Storage_Type  Storage_GB  Touchscreen  Weight_kg  Screen_Size_inch  Discount_percent  Price_SGD  Brand_Discount  Member_Discount
          0  Acer            Aspire 5          Intel i9-14900HK NVIDIA RTX 4070      64          SSD         256        False       1.56              16.0              5.28    3207.60              80           160.38
          1  Acer             Nitro 5        AMD Ryzen 9 8900HX AMD Radeon 780M      32          SSD        1024         True       1.45              14.0              6.01    2568.40              80           179.79
          2  Acer             Nitro 5         AMD Ryzen 5 8600H NVIDIA 

# <font color='red'>**Chapter 8. Ask the LLM to review data quality using payload text**


In [16]:
prompt_quality = f"""
You are an expert data scientist with extensive knowledge of tree-based models.
Always justify recommendations using reasoning trace based ONLY on the dataset profile.
============================
DATASET PROFILE
============================
{payload_text}

============================
TASK
============================
Assess the dataset's quality and modelling readiness.

1. Identify all confirmed data quality or modelling-readiness issues.

For each issue, provide:
- Issue
- Column(s) affected
- Evidence from the dataset profile
- Why it may affect analysis or modelling
- Recommended action
- Priority (High / Medium / Low)

2. Identify any potential issues that require further investigation but cannot be confirmed from the dataset profile alone.

3. Highlight the strengths of the dataset (for example, no missing values, no duplicates, appropriate data types, etc.).

4. State whether the dataset is ready for machine learning modelling using ONE of the following ratings:
- Ready
- Ready with Minor Cleaning
- Ready with Moderate Cleaning
- Not Ready
Provide a brief justification.

5. Recommend the Top 3 next steps before modelling.

6. Provide a python script to handle the identified issues.
- Define one helper function for each issue.
- Then define a wrapper function that calls these helper with true false option as user choice
- Provide a single line of code to run the overall wrapper function.
- Do not encode categorical columns or model first.

========================
IMPORTANT INSTRUCTIONS
========================

- Use only the information contained in the dataset profile.
- Do NOT assume information that is not provided.
- If evidence is unavailable, state "Not available" instead of guessing.
- Reference the relevant dataset profile section whenever possible (e.g., Missing Values, Numeric Summary, Correlation Matrix, Warning Checks).
- Do NOT recommend dropping a column solely because it appears in a warning.
- Treat ID-like, high-cardinality, constant, near-constant and outlier warnings as items requiring investigation, not automatic removal.
- Do NOT treat correlation as proof of causation.
- Do NOT recommend removing outliers unless there is evidence they are invalid or unsuitable for the business objective.
- If skewness, outlier summary, feature importance, or other analyses are not included in the dataset profile, state that they were not available instead of inferring results.
- Distinguish clearly between confirmed issues and potential issues.
- Keep recommendations practical, concise, and suitable for a data analytics student.

========================
OUTPUT FORMAT
========================

Overall Modelling Readiness
---------------------------
Readiness Rating:
Brief Justification:

Confirmed Issues
----------------
For each issue, use the following format:

Issue:
Column(s):
Evidence:
Impact:
Recommended Action:
Priority:

Potential Issues Requiring Investigation
-----------------------------------------
For each potential issue, use:

Issue:
Reason Further Investigation is Needed:

Dataset Strengths
-----------------
- ...
- ...
- ...

Checks with No Issues
---------------------
- ...
- ...
- ...

Top 3 Recommended Next Steps
----------------------------
1.
2.
3.

Overall Summary
---------------
Provide a concise summary (3–5 sentences) describing the dataset's overall quality and readiness for machine learning modelling.
"""

if client is not None:
    response_quality= client.responses.create(
        model=OPENAI_MODEL,
        input=prompt_quality
    )
    print('\n=== LLM response ===')
    print(response_quality.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')

# This cell asks the LLM to review the dataset profile for modelling-readiness issues.
# The LLM does not receive the full dataset, only the summary stored in payload_text.
# The prompt asks for possible issues and practical remedies.
# The instructions help prevent the LLM from making unsupported assumptions or dropping columns too quickly.
# If the API client is connected, the prompt is sent automatically.
# Otherwise, students can copy the prompt and use the chatbot manually.


=== LLM response ===
Overall Modelling Readiness
---------------------------
Readiness Rating: **Ready with Minor Cleaning**

Brief Justification:
- The dataset has **no missing values** and **no duplicate rows**, and data types largely look suitable for ML.  
- However, there is a **confirmed leakage risk** from an ID-like column (**`Unnamed: 0`**), and a **near-constant/low-information feature** exists (**`Storage_Type` is constant**).  
- Other concerns (e.g., categorical encoding needs, potential target leakage via discounts) are **not confirmed** from the profile and require investigation, so this is not “Ready” without any cleaning.

Confirmed Issues
----------------
### 1) Potential target leakage / ID-like feature
**Issue:** ID-like column likely to capture row identity rather than laptop properties.  
**Column(s):** `Unnamed: 0`  
**Evidence:**  
- **Possible leakage columns (unique for each row):** `['Unnamed: 0']`  
- **Unique values:** `Unnamed: 0` has **1000 unique** valu

#<font color='red'>**COPY the Output from above and output Python codes to clean your dataset**

In [26]:
prompt_clean = f"""
You are an expert data scientist with experience in tree-based regression models.
Generate executable python code to fix each of the following issues, only for data cleaning purposes, do not model yet.
Input is df, and output should be cleaned_df.
Issues:
{response_quality.output_text}

Additionally, generate code for:
- safe column checks
- Extreme sparsity / large cardinality in categoricals
- Robust outlier clipping (only for plausibly bounded numeric features)
- Apply clipping to numeric predictors only (exclude target if present)
- ensure no accidental all-null columns

Lastly, show the columns of df and cleaned_df, to compare the differences, i.e. which columns are added or removed

"""


if client is not None:
    response_clean = client.responses.create(
        model=OPENAI_MODEL,
        input=prompt_clean
    )
    print('\nLLM response:')
    print(response_clean.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')


LLM response:
```python
import pandas as pd
import numpy as np

# ----------------------------
# Helpers: safe checks + cleaning
# ----------------------------

def safe_drop_columns(df: pd.DataFrame, cols) -> pd.DataFrame:
    cols = [c for c in cols if c in df.columns]
    return df.drop(columns=cols, errors="ignore")

def drop_constant_columns(df: pd.DataFrame, threshold_unique: int = 1) -> pd.DataFrame:
    """
    Drops columns with <= threshold_unique unique values (including NaN if present).
    """
    nunique = df.nunique(dropna=False)
    constant_cols = nunique[nunique <= threshold_unique].index.tolist()
    return df.drop(columns=constant_cols, errors="ignore")

def ensure_no_all_null_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drops columns that are entirely null.
    """
    all_null_cols = [c for c in df.columns if df[c].isna().all()]
    return df.drop(columns=all_null_cols, errors="ignore")

def clip_numeric_predictors(
    df: pd.DataFrame,
    target_col:

# **Compare df and cleaned_df after cleaning**

In [29]:
# What to check after cleaning:
# 1. Did the number of rows stay the same?
# 2. Did the discount columns disappear?
# 3. Are there missing values that still need attention?
# 4. Does Price_SGD still exist as the target column?

print('Original rows:', df.shape[0])
print('Cleaned rows:', cleaned_df.shape[0])

print('\nMissing values after cleaning:')
missing_after_cleaning = cleaned_df.isna().sum()
display(missing_after_cleaning.sort_values(ascending=False).head(10))

print('\nColumns after cleaning:')
print(cleaned_df.columns.tolist())

Original rows: 1000
Cleaned rows: 1000

Missing values after cleaning:


,0
Brand,0
Model,0
CPU,0
GPU,0
RAM_GB,0
Storage_GB,0
Touchscreen,0
Weight_kg,0
Screen_Size_inch,0
Discount_percent,0



Columns after cleaning:
['Brand', 'Model', 'CPU', 'GPU', 'RAM_GB', 'Storage_GB', 'Touchscreen', 'Weight_kg', 'Screen_Size_inch', 'Discount_percent', 'Price_SGD', 'Brand_Discount', 'Member_Discount']


#**Save your cleaned_df**

In [30]:
#aved to Google colab /content. Files in /content are temporary and disappear when the Colab runtime ends
cleaned_df.to_csv('cleaned_laptop_prices.csv', index=False)

print('Saved cleaned dataset as cleaned_laptop_prices.csv')
print('You can use this file in Session 2.')

import os
print(os.getcwd())
print(os.listdir())

Saved cleaned dataset as cleaned_laptop_prices.csv
You can use this file in Session 2.
/content
['.config', 'cleaned_laptop_prices.csv', 'sample_data']


In [31]:
# Saved the file in Google > MyDrive

from google.colab import drive
drive.mount('/content/drive')

cleaned_df.to_csv(
    "/content/drive/MyDrive/cleaned_laptop_prices.csv",
    index=False
)

Mounted at /content/drive


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#Saved the file in my PC "Downloads" folder

from google.colab import files
files.download("cleaned_laptop_prices.csv")


#**Section 2: From cleaned dataset to baseline model**

#**Chapter 12. Load cleaned_df**

In [33]:
# You may run the following code, if you have the cleaned_laptop_prices.csv file. If not, just rerun earlier cells to re-create it.
#cleaned_df = pd.read_csv('cleaned_laptop_prices.csv')

# Or read the cleaned_laptop_prices.csv  file from Google Drive
# cleaned_df = pd.read_csv("/content/drive/MyDrive/cleaned_laptop_prices.csv")

# read from github
github_raw_url = 'https://raw.githubusercontent.com/zawhtetwai/ADALL_GitHub/refs/heads/main/cleaned_laptop_prices.csv'
cleaned_df = pd.read_csv(github_raw_url)

print('Loaded cleaned_laptop_prices.csv.')
print('Cleaned shape:', cleaned_df.shape)
display(cleaned_df.head())

Loaded cleaned_laptop_prices.csv.
Cleaned shape: (1000, 10)


,Brand,Model,CPU,GPU,RAM_GB,Storage_GB,Touchscreen,Weight_kg,Screen_Size_inch,Price_SGD
0,Acer,Aspire 5,Intel i9-14900HK,NVIDIA RTX 4070,64,256,False,1.56,16.0,3640.18
1,Acer,Nitro 5,AMD Ryzen 9 8900HX,AMD Radeon 780M,32,1024,True,1.45,14.0,3009.03
2,Acer,Nitro 5,AMD Ryzen 5 8600H,NVIDIA RTX 4050,32,2048,False,1.34,14.0,2434.03
3,Acer,TravelMate P6,Intel Core Ultra 7 15500H,NVIDIA RTX 4060,16,4096,True,1.18,13.3,2863.30
4,Acer,Predator Helios 300,Intel i7-14800H,NVIDIA RTX 4070,8,1024,True,1.31,14.0,3036.30


#**Chapter 14. Split into X and y**
- X - Input features used to predict target. It should not contain target column
- y - actual target

In [36]:
target_col = 'Price_SGD'

if target_col not in cleaned_df.columns:
    raise ValueError(f'Target column {target_col} was not found. Check your cleaned dataset columns.')

X = cleaned_df.drop(columns=[target_col])
y = cleaned_df[target_col]

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Target column:', target_col)
display(X.head())
display(y.head())

# This cell separates the dataset into INPUT features X and OUTPUT target y.
# target_col stores the column we want the model to predict, which is Price_SGD.
# The if-statement checks that Price_SGD still exists after data cleaning.
# X contains all columns except the target column.
# y contains only the target column.
# The shape outputs help us check how many rows and columns are being used for modelling.

X shape: (1000, 9)
y shape: (1000,)
Target column: Price_SGD


,Brand,Model,CPU,GPU,RAM_GB,Storage_GB,Touchscreen,Weight_kg,Screen_Size_inch
0,Acer,Aspire 5,Intel i9-14900HK,NVIDIA RTX 4070,64,256,False,1.56,16.0
1,Acer,Nitro 5,AMD Ryzen 9 8900HX,AMD Radeon 780M,32,1024,True,1.45,14.0
2,Acer,Nitro 5,AMD Ryzen 5 8600H,NVIDIA RTX 4050,32,2048,False,1.34,14.0
3,Acer,TravelMate P6,Intel Core Ultra 7 15500H,NVIDIA RTX 4060,16,4096,True,1.18,13.3
4,Acer,Predator Helios 300,Intel i7-14800H,NVIDIA RTX 4070,8,1024,True,1.31,14.0


,Price_SGD
0,3640.18
1,3009.03
2,2434.03
3,2863.30
4,3036.30


#**Chapter 15. Train-Test split**

In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2, #0.2 = 20%
    random_state=42
)

print('Training rows:', X_train.shape[0])
print('Test rows:', X_test.shape[0])

#value count for y_train and y_test
#print(y_train.value_counts(normalize=True))
#print(y_test.value_counts(normalize=True))

Training rows: 800
Test rows: 200


#**Chapter 16. Identify numeric and categorical columns**
- Numeric column - Scale using StandardScaler
- Categorical	columns - Convert using OneHotEncoder

In [38]:
cat_columns = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_columns = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

print('Categorical columns:', cat_columns)
print('Numeric columns:', num_columns)
# This cell separates the input columns by data type.
# cat_columns stores text/category columns, such as brand or CPU type.
# num_columns stores numeric columns, such as RAM, storage, or weight.
# This is needed because categorical and numeric columns usually need different preprocessing steps.

Categorical columns: ['Brand', 'Model', 'CPU', 'GPU']
Numeric columns: ['RAM_GB', 'Storage_GB', 'Touchscreen', 'Weight_kg', 'Screen_Size_inch']


#**Chapter 17. Build the preprocessing pipeline**

In [45]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_columns),
        ('num', StandardScaler(), num_columns)
        # ('num', 'passthrough', num_columns) -- use this if numeric columns are passed through without any change. Usually this is better for tree models
    ],
    remainder='drop'
)

preprocessor

# This cell creates the preprocessing steps before modelling.
# OneHotEncoder changes categorical columns into numeric 0/1 columns.
# handle_unknown='ignore' prevents errors if new unseen categories appear later during testing/deployment.
# StandardScaler scales numeric columns so they are on a more similar range.
# ColumnTransformer applies the correct preprocessing to each column group.
# remainder='drop' means columns not listed in cat_columns or num_columns will be removed.

ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['Brand', 'Model', 'CPU', 'GPU']),
                                ('num', StandardScaler(),
                                 ['RAM_GB', 'Storage_GB', 'Touchscreen',
                                  'Weight_kg', 'Screen_Size_inch'])])

#**Chapter 18. Train a simple baseline model**

In [46]:
baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=42, max_depth=12))
])

baseline_model.fit(X_train, y_train)
print('Baseline model trained.')

Baseline model trained.


#**Chapter 19. Evaluate the baseline model**
| Metric | Plain meaning | Better when |
|---|---|---|
| MAE | Average absolute error in dollars | Lower is better |
| RMSE | Error that punishes large mistakes more | Lower is better |
| R² | How much variation is explained | Higher is better |

For business explanation, MAE is often easiest because it is in the same unit as the target: Singapore dollars.

In [47]:
y_pred = baseline_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

baseline_results = pd.DataFrame({
    'Model': ['Baseline Decision Tree'],
    'MAE': [mae],
    'RMSE': [rmse],
    'R2': [r2]
})

display(baseline_results.round(3))

# This cell uses the trained baseline model to predict laptop prices on the test set.
# y_pred stores the predicted prices.
# MAE shows the average absolute prediction error in dollars.
# RMSE also measures prediction error, but gives more penalty to large errors.
# R2 shows how much of the price variation is explained by the model.
# baseline_results stores the metrics in a simple table for comparison later.

,Model,MAE,RMSE,R2
0,Baseline Decision Tree,102.487,146.177,0.961


#<font color='red'>**Chapter 19b. Explain the metrics, preprocessor, model and pipeline**

In [51]:
prompt_evaluate = f"""
You are an expert data scientist with experience in tree-based regression models.
Based on the baseline model results {baseline_results} where target is in SGD
Explain the metrics in simple terms

"""
if client is not None:
    response_evaluate = client.responses.create(
        model=OPENAI_MODEL,
        input=prompt_evaluate
    )
    print('\n=====LLM response=====')
    print(response_evaluate.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')


LLM response:
Sure—here’s what those metrics mean in simple terms for your regression task (predicting the target in **SGD**).

### **1) MAE (Mean Absolute Error): 102.49**
- **What it tells you:** The average absolute difference between your model’s predictions and the true values.
- **In practice:** On average, the model is off by about **102.49 SGD** per prediction (regardless of whether it predicts too high or too low).

### **2) RMSE (Root Mean Squared Error): 146.18**
- **What it tells you:** Similar to MAE, but it penalizes larger errors more heavily (because it squares errors before averaging).
- **In practice:** Typical prediction error magnitude is about **146.18 SGD**, and this value suggests that there are **some bigger mistakes** pulling the error upward.

### **3) R² (Coefficient of Determination): 0.9611**
- **What it tells you:** How much of the variation in the actual target values your model explains.
- **In practice:** An **R² of 0.961** means the model explains abo

In [52]:
prompt_explain = f"""
You are an expert data scientist with experience in tree-based regression models.
Based on the model {baseline_model} and preprocessor {preprocessor}
Explain in simple terms:
- What the preprocessor does
- What the model does
- Why a pipeline prevents mistakes
"""
if client is not None:
    response_explain = client.responses.create(
        model=OPENAI_MODEL,
        input=prompt_explain
    )
    print('\n=====LLM response=====')
    print(response_explain.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')


=====LLM response=====
### 1) What the **preprocessor** does
The preprocessor prepares your features so the regression model can learn from them.

- **Categorical columns**: `Brand`, `Model`, `CPU`, `GPU`  
  - It uses **OneHotEncoder** to convert each category into its own binary column (e.g., “Brand=Apple” becomes 1/0).  
  - `handle_unknown='ignore'` means: if you later see a brand/CPU/model value during prediction that wasn’t in training, it won’t crash—it will just produce all-zeros for unseen categories.

- **Numerical columns**: `RAM_GB`, `Storage_GB`, `Touchscreen`, `Weight_kg`, `Screen_Size_inch`  
  - It applies **StandardScaler**, which rescales these numbers to have **mean 0 and standard deviation 1**.  
  - This helps keep numeric features on comparable scales.

- Then it **combines** the encoded categorical features + the scaled numeric features into a single matrix that the model can use.

---

### 2) What the **model** does
The model is a **DecisionTreeRegressor** with